# Geopack SDK: Data Management and Task Orchestration

This notebook demonstrates advanced data management features: uploading new files, exporting existing datasets to various formats, and monitoring background tasks.

---

### 🚀 Setup Note
This notebook is configured to work both with an installed `geopack-sdk` package or directly from the repository source code (`src/` folder).

---

In [ ]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv import load_dotenv

# --- SMART SOURCE IMPORT ---
try:
    current_dir = os.getcwd()
    source_path = os.path.abspath(os.path.join(current_dir, "..", "src"))
    if os.path.exists(source_path):
        if source_path not in sys.path:
            sys.path.insert(0, source_path)
        print(f"ℹ️ Using SDK from local source: {source_path}")
    else:
        print("ℹ️ Using SDK from installed site-packages (pip)")
except Exception:
    print("⚠️ Could not determine local source path, falling back to pip.")

from geopack_sdk import GeopackClient

load_dotenv()
client = GeopackClient(base_url=os.getenv("GEOPACK_API_URL", "http://localhost:3000/api"))

# Login using credentials from .env or defaults
try:
    client.auth.login(
        os.getenv("GEOPACK_USERNAME", "admin"), 
        os.getenv("GEOPACK_PASSWORD", "password")
    )
    print("✅ Login successful!")
except Exception as e:
    print(f"❌ Login failed: {e}")


ℹ️ Using SDK from local source: d:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk\src
✅ Login successful!


## 1. Environment Preparation
Before uploading, we need to choose a **DataStore** (where the data will be stored) and a **Workgroup** (who will own the data).

In [ ]:
# 1. List available DataStores
datastores = client.datastores.list()
print("Available DataStores:")
for ds in datastores:
    #print(f"- [{ds['id']}] {ds['name']} ({ds['type']})")
    print(f"- [{ds['id']}] {ds['name']}  ({ds.get('dataType', 'N/A')} {ds.get('subType', '')}) [{ds.get('dataStore', {}).get('name', 'N/A')}]")


# 2. List available Workgroups
workgroups = client.workgroups.list()
print("\nAvailable Workgroups:")
for wg in workgroups:
    print(f"- [{wg['id']}] {wg['name']}")

# --- SET YOUR TARGETS HERE ---
# By default, we'll use the first ones found if you don't change these
SELECTED_DATASTORE_ID = datastores[0]['id'] if datastores else 1
SELECTED_WORKGROUP_ID = workgroups[0]['id'] if workgroups else 1

print(f"\n✅ Ready to use: DataStore ID={SELECTED_DATASTORE_ID}, Workgroup ID={SELECTED_WORKGROUP_ID}")

Available DataStores:
- [11] Default Filesystem GDB (filesystem)
- [9] Default PostgreSQL GDB (postgres)
- [10] Default SQL Server GDB (mssql)
- [31] GDB_Golbahar (esri-geodatabase-mssql)
- [13] pg_test (postgres)
- [32] pg3 (postgres)
- [27] sql123 (mssql)

Available Workgroups:
- [1] Default Workgroup
- [4] gdb-kr-postgres
- [6] multi-criteria-overlay
- [5] Rigan bam
- [3] w2

✅ Ready to use: DataStore ID=11, Workgroup ID=1


## 1. Uploading a New Dataset
The SDK handles the two-step upload process: temporary file upload followed by background processing.

In [6]:
# Path to a local GeoJSON or Shapefile
test_file = "data/Rural_District.shp.geojson" 

if os.path.exists(test_file):
    print(f"Uploading {test_file}...")
    # Using the IDs selected in the previous step
    task_result = client.datasets.upload(
        file_path=test_file, 
        data_store_id=SELECTED_DATASTORE_ID, 
        workgroup_id=SELECTED_WORKGROUP_ID, 
        wait=True
    )
    
    # Extract results from the task output
    results = task_result.get("results", [])
    
    print("\n✅ Upload successful! New datasets created:")
    if results:
        for ds in results:
            print(f"- {ds['datasetName']} (ID: {ds['createdDatasetId']})")
    else:
        print("! Task finished but results field is empty.")
else:
    print(f"⚠️ File '{test_file}' not found. Please ensure the file exists in the 'data' folder.")


Uploading data/Rural_District.shp.geojson...

✅ Upload successful! New datasets created:
- Rural_District.shp (ID: 2362)


## 2. Exporting and Downloading Data
Need to download a dataset in a specific format? Geopack orchestrates an export task.

In [7]:
# Let's export the first available dataset to GeoPackage
datasets = client.datasets.list(page_size=1)
if datasets:
    ds_id = datasets[0]['id']
    print(f"Exporting Dataset #{ds_id} to GPKG...")
    
    # 1. Start export and wait for completion
    task_result = client.datasets.export(dataset_id=ds_id, workgroup_id=1, format='gpkg', wait=True)
    
    # 2. Download the resulting file
    os.makedirs("downloads", exist_ok=True)
    local_file = client.datasets.download(task_result, "downloads/")
    
    file_size = os.path.getsize(local_file) / (1024 * 1024)
    print(f"✅ Downloaded: {local_file} ({file_size:.2f} MB)")

Exporting Dataset #2362 to GPKG...
Waiting for task 86b7ac34-9726-4229-ad70-dd1372c32303 to complete...
  Task 86b7ac34-9726-4229-ad70-dd1372c32303: processing
  Task 86b7ac34-9726-4229-ad70-dd1372c32303: completed
✅ Downloaded: d:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk\notebooks\downloads\Rural_District_shp.gpkg (2.42 MB)


## 3. Direct Task Monitoring
You can also monitor any background task manually using its ID.

In [8]:
# Example: Getting status of a specific task
if 'taskId' in locals():
    status = client.tasks.get_status(taskId)
    print(f"Manual check for task {taskId}: {status['status']}")